# 🏥 Sistema Preditivo de Obesidade + Dashboard Analítico
### Tech Challenge – Fase 04 | POSTECH Data Analytics

Este notebook é uma **versão explicativa e exploratória** do aplicativo `app.py`, originalmente construído em **Streamlit**.

## 📌 Sobre o projeto

O objetivo da aplicação é apoiar uma equipe médica na **avaliação do risco de obesidade** de pacientes, combinando:

1. **Sistema Preditivo** — um modelo de Machine Learning (LightGBM, com fallback treinado automaticamente caso não haja um modelo salvo) que recebe dados antropométricos, hábitos alimentares e estilo de vida do paciente e prediz o **nível de obesidade** (de "Peso Insuficiente" até "Obesidade Tipo III").
2. **Dashboard Analítico** — um conjunto de visualizações (Plotly) que exploram a base de dados `Obesity.csv`, revelando relações entre hábitos (atividade física, alimentação, transporte, histórico familiar, etc.) e o nível de obesidade, para apoiar decisões clínicas e de saúde pública.

## 🗂️ Base de dados

O projeto espera um arquivo `Obesity.csv` com colunas como:

| Coluna | Significado |
|---|---|
| `Gender`, `Age`, `Height`, `Weight` | Dados demográficos/antropométricos |
| `family_history` | Histórico familiar de excesso de peso (yes/no) |
| `FAVC` | Consumo frequente de alimentos calóricos (yes/no) |
| `FCVC` | Frequência de consumo de vegetais (1–3) |
| `NCP` | Número de refeições principais por dia |
| `CAEC` | Come entre as refeições (no/Sometimes/Frequently/Always) |
| `SMOKE` | Fumante (yes/no) |
| `CH2O` | Consumo diário de água (1–3) |
| `SCC` | Monitora ingestão calórica (yes/no) |
| `FAF` | Frequência de atividade física (0–3) |
| `TUE` | Tempo em dispositivos eletrônicos (0–2) |
| `CALC` | Frequência de consumo de álcool |
| `MTRANS` | Meio de transporte habitual |
| `Obesity` | **Variável-alvo**: nível de obesidade (7 classes) |

## ⚙️ Tecnologias utilizadas

- **pandas / numpy** — manipulação de dados
- **scikit-learn** — pré-processamento (encoding, escalonamento)
- **LightGBM** — modelo preditivo (classificação multiclasse)
- **Plotly** — visualizações interativas
- **Streamlit** — framework original da interface web (aqui adaptado para notebook)

> ⚠️ **Nota sobre a conversão**: o `app.py` original é uma aplicação **Streamlit** (`streamlit run app.py`), pensada para rodar como página web interativa, com formulários, sidebar e navegação entre páginas. Um notebook não possui esses componentes de interface. Por isso, aqui:
> - os comandos `st.*` foram removidos ou substituídos por equivalentes de notebook (`display()`, `fig.show()`, `print()`);
> - o formulário de entrada do paciente foi substituído por um **dicionário de exemplo editável** (célula de código), que você pode alterar livremente para simular diferentes pacientes;
> - a lógica de negócio (carregamento de dados, engenharia de atributos, treinamento do modelo, predição e gráficos) foi **100% preservada**.


## 1. Importações

Bibliotecas utilizadas para manipulação de dados, modelagem e visualização.


In [ ]:
import os
import pickle
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, Markdown, HTML

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 50)


## 2. Paleta de cores e constantes do projeto

O projeto usa **três paletas de cores** para diferentes contextos visuais:

- `COLOR_MAP`: escala de verdes (do mais claro ao mais escuro), usada nos gráficos "neutros" do dashboard.
- `COLOR_MAP_DASH`: escala semafórica (verde → amarelo → vermelho), usada para reforçar a gravidade do nível de obesidade.
- `COLOR_MAP_STEPS`: variação usada no gauge de IMC e em gráficos de "steps".

Além disso, são definidos:
- `ORDER`: a ordem lógica das 7 classes de obesidade (da mais leve à mais grave).
- `LABEL_PT`: tradução das classes para rótulos em português.
- `RECOMMENDATIONS`: recomendações clínicas específicas para cada nível de obesidade.


In [ ]:
COLOR_MAP = {
    "Insufficient_Weight":  "#c8e6c9",
    "Normal_Weight":        "#a5d6a7",
    "Overweight_Level_I":   "#81c784",
    "Overweight_Level_II":  "#66bb6a",
    "Obesity_Type_I":       "#43a047",
    "Obesity_Type_II":      "#2e7d32",
    "Obesity_Type_III":     "#1b5e20",
}

COLOR_MAP_DASH = {
    "Insufficient_Weight":  "#81c784",
    "Normal_Weight":        "#aed581",
    "Overweight_Level_I":   "#fff176",
    "Overweight_Level_II":  "#ffb74d",
    "Obesity_Type_I":       "#ff8a65",
    "Obesity_Type_II":      "#ef5350",
    "Obesity_Type_III":     "#c62828",
}

COLOR_MAP_STEPS = {
    "Insufficient_Weight": "#acd6ae",
    "Normal_Weight":       "#c8df7a",
    "Overweight_Level_I":  "#fff38a",
    "Overweight_Level_II": "#ffc96b",
    "Obesity_Type_I":      "#ffa285",
    "Obesity_Type_II":     "#ec8482",
    "Obesity_Type_III":    "#db6666",
}

# Cores de destaque para gênero
COLOR_F = "#FD3DB5"
COLOR_M = "#0000FF"

ORDER = [
    "Insufficient_Weight", "Normal_Weight",
    "Overweight_Level_I", "Overweight_Level_II",
    "Obesity_Type_I", "Obesity_Type_II", "Obesity_Type_III",
]

LABEL_PT = {
    "Insufficient_Weight":  "Abaixo do Peso",
    "Normal_Weight":        "Peso Normal",
    "Overweight_Level_I":   "Sobrepeso Grau I",
    "Overweight_Level_II":  "Sobrepeso Grau II",
    "Obesity_Type_I":       "Obesidade Tipo I",
    "Obesity_Type_II":      "Obesidade Tipo II",
    "Obesity_Type_III":     "Obesidade Tipo III",
}

RECOMMENDATIONS = {
    "Insufficient_Weight": [
        "Consultar nutricionista para plano de ganho de peso saudável.",
        "Aumentar ingestão calórica com alimentos nutritivos.",
        "Avaliar causas subjacentes (distúrbios alimentares, hipertireoidismo).",
        "Praticar musculação para ganho de massa magra.",
    ],
    "Normal_Weight": [
        "Manter hábitos atuais de alimentação e atividade física.",
        "Realizar check-up anual preventivo.",
        "Monitorar IMC periodicamente.",
        "Continuar com hidratação adequada (≥ 2L/dia).",
    ],
    "Overweight_Level_I": [
        "Reduzir consumo de alimentos ultraprocessados e calóricos.",
        "Iniciar prática regular de atividade física (3×/semana).",
        "Monitorar ingestão calórica diária.",
        "Consultar médico para avaliação cardiometabólica.",
    ],
    "Overweight_Level_II": [
        "Intervenção nutricional urgente com acompanhamento profissional.",
        "Aumentar frequência de atividade física para ≥ 4×/semana.",
        "Reduzir consumo de álcool e alimentos calóricos.",
        "Avaliar risco de síndrome metabólica, hipertensão e diabetes.",
    ],
    "Obesity_Type_I": [
        "Encaminhar para equipe multidisciplinar (médico, nutricionista, psicólogo).",
        "Programa estruturado de perda de peso (déficit calórico controlado).",
        "Exames: glicemia, perfil lipídico, pressão arterial.",
        "Considerar atividade física de baixo impacto (natação, caminhada).",
    ],
    "Obesity_Type_II": [
        "Avaliação médica imediata para comorbidades associadas.",
        "Considerar tratamento farmacológico adjuvante.",
        "Programa intensivo de mudança de estilo de vida.",
        "Monitoramento contínuo de pressão arterial, glicemia e função hepática.",
    ],
    "Obesity_Type_III": [
        "Avaliação para cirurgia bariátrica (IMC ≥ 40).",
        "Acompanhamento médico intensivo e multidisciplinar.",
        "Rastreamento de apneia do sono, diabetes tipo 2, doenças cardiovasculares.",
        "Suporte psicológico e grupos de apoio.",
    ],
}


def fmt_num(n: int) -> str:
    """Formata inteiro com separador de milhar por ponto (padrão brasileiro)."""
    return f"{n:,}".replace(",", ".")


## 3. Carregamento dos dados

A função `load_data` procura o arquivo `Obesity.csv` em alguns caminhos possíveis.
No app original, essa função usava `@st.cache_data` para evitar recarregar os dados a cada interação;
em um notebook isso não é necessário, então essa decoração foi removida.

> 📁 **Para executar esta célula**, coloque o arquivo `Obesity.csv` na mesma pasta deste notebook (ou em uma subpasta `data/`).


In [ ]:
def load_data():
    paths = ["Obesity.csv", "data/Obesity.csv"]
    for p in paths:
        if os.path.exists(p):
            return pd.read_csv(p)
    raise FileNotFoundError(
        "❌ Arquivo Obesity.csv não encontrado. "
        "Coloque-o na mesma pasta deste notebook (ou em 'data/Obesity.csv')."
    )


try:
    df_raw = load_data()
    print(f"✅ Dados carregados: {fmt_num(len(df_raw))} registros, {df_raw.shape[1]} colunas.")
    display(df_raw.head())
except FileNotFoundError as e:
    print(e)
    df_raw = None


## 4. Engenharia de atributos e modelo preditivo

A função `engineer_features` transforma os dados brutos do paciente exatamente da mesma forma usada no treinamento do modelo, garantindo consistência entre treino e predição. As principais transformações são:

- **IMC (`BMI`)**: calculado como `Weight / Height²`.
- **`risk_score`**: pontuação de fatores de risco (consumo calórico frequente, histórico familiar, tabagismo, não monitorar calorias, comer entre refeições com frequência, consumo de álcool).
- **`healthy_score`**: pontuação de hábitos saudáveis (atividade física, consumo de vegetais, hidratação, monitoramento calórico, transporte ativo).
- **Codificação categórica**: variáveis binárias (`Gender`, `family_history`, `FAVC`, `SMOKE`, `SCC`) viram 0/1; `CAEC` e `CALC` viram escalas ordinais (0 a 3); `MTRANS` vira *one-hot encoding*.

Se não existir um modelo salvo em `model/` (`best_model.pkl`, `scaler.pkl`, `label_encoder.pkl`, `metadata.pkl`), a função `_train_fallback_model` treina um **LightGBM** do zero, usando os próprios dados de `Obesity.csv`.


In [ ]:
def engineer_features(input_df: pd.DataFrame, feature_columns: list) -> pd.DataFrame:
    """Aplica a mesma engenharia de atributos usada no treinamento do modelo."""
    df = input_df.copy()
    for col in ["FCVC", "NCP", "CH2O", "FAF", "TUE"]:
        if col in df.columns:
            df[col] = df[col].round().astype(int)
    df["BMI"] = df["Weight"] / (df["Height"] ** 2)
    df["risk_score"] = (
        (df["FAVC"] == "yes").astype(int) * 2 +
        (df["family_history"] == "yes").astype(int) * 2 +
        (df["SMOKE"] == "yes").astype(int) +
        (df["SCC"] == "no").astype(int) +
        (df["CAEC"] == "Always").astype(int) * 2 +
        (df["CAEC"] == "Frequently").astype(int) +
        (df["CALC"] == "Always").astype(int) * 2 +
        (df["CALC"] == "Frequently").astype(int)
    )
    df["healthy_score"] = (
        (df["FAF"] >= 2).astype(int) * 2 +
        (df["FCVC"] >= 2).astype(int) +
        (df["CH2O"] >= 2).astype(int) +
        (df["SCC"] == "yes").astype(int) +
        df["MTRANS"].isin(["Walking", "Bike"]).astype(int)
    )
    for col in ["Gender", "family_history", "FAVC", "SMOKE", "SCC"]:
        df[col] = df[col].map({"Male": 1, "Female": 0, "yes": 1, "no": 0}).fillna(0).astype(int)
    df["CAEC"] = df["CAEC"].map({"no": 0, "Sometimes": 1, "Frequently": 2, "Always": 3})
    df["CALC"] = df["CALC"].map({"no": 0, "Sometimes": 1, "Frequently": 2, "Always": 3})
    df = pd.get_dummies(df, columns=["MTRANS"], drop_first=False)
    for col in feature_columns:
        if col not in df.columns:
            df[col] = 0
    return df[feature_columns]


def _train_fallback_model(df_raw: pd.DataFrame):
    """Treina um LightGBM do zero a partir de Obesity.csv, caso não haja modelo salvo."""
    import lightgbm as lgb
    from sklearn.preprocessing import LabelEncoder, StandardScaler

    dw = df_raw.copy()
    for col in ["FCVC", "NCP", "CH2O", "FAF", "TUE"]:
        dw[col] = dw[col].round().astype(int)
    dw["BMI"] = dw["Weight"] / (dw["Height"] ** 2)  # cria coluna BMI (IMC)
    dw["risk_score"] = (  # engenharia de atributos: pontuação de hábitos de risco
        (dw["FAVC"] == "yes").astype(int) * 2 +
        (dw["family_history"] == "yes").astype(int) * 2 +
        (dw["SMOKE"] == "yes").astype(int) +
        (dw["SCC"] == "no").astype(int) +
        (dw["CAEC"] == "Always").astype(int) * 2 +
        (dw["CAEC"] == "Frequently").astype(int) +
        (dw["CALC"] == "Always").astype(int) * 2 +
        (dw["CALC"] == "Frequently").astype(int)
    )
    dw["healthy_score"] = (  # conta hábitos saudáveis
        (dw["FAF"] >= 2).astype(int) * 2 +
        (dw["FCVC"] >= 2).astype(int) +
        (dw["CH2O"] >= 2).astype(int) +
        (dw["SCC"] == "yes").astype(int) +
        dw["MTRANS"].isin(["Walking", "Bike"]).astype(int)
    )
    for col in ["Gender", "family_history", "FAVC", "SMOKE", "SCC"]:  # transforma texto em números
        dw[col] = dw[col].map({"Male": 1, "Female": 0, "yes": 1, "no": 0}).fillna(0).astype(int)
    dw["CAEC"] = dw["CAEC"].map({"no": 0, "Sometimes": 1, "Frequently": 2, "Always": 3})
    dw["CALC"] = dw["CALC"].map({"no": 0, "Sometimes": 1, "Frequently": 2, "Always": 3})
    dw = pd.get_dummies(dw, columns=["MTRANS"], drop_first=False)

    label_order = ORDER  # define a ordem das classes
    le = LabelEncoder()
    le.fit(label_order)  # aprende a correspondência ex: Insufficient_Weight: 0, Normal_Weight: 1...
    X = dw.drop(columns=["Obesity"])
    y = le.transform(dw["Obesity"])
    scaler = StandardScaler()
    X_sc = scaler.fit_transform(X)
    model = lgb.LGBMClassifier(n_estimators=200, learning_rate=0.1, random_state=42, verbose=-1)
    model.fit(X_sc, y)
    meta = {
        "best_model_name": "LightGBM (auto-treinado)",
        "feature_columns": list(X.columns),
        "scaled": True,
        "accuracy": None,
    }
    return model, scaler, le, meta


def load_model(df_raw: pd.DataFrame):
    """Carrega modelo salvo em disco, ou treina um modelo fallback caso não exista."""
    model_dir = "model"
    required = ["best_model.pkl", "scaler.pkl", "label_encoder.pkl", "metadata.pkl"]
    if all(os.path.exists(os.path.join(model_dir, f)) for f in required):
        with open(f"{model_dir}/best_model.pkl", "rb") as f: model = pickle.load(f)
        with open(f"{model_dir}/scaler.pkl", "rb") as f: scaler = pickle.load(f)
        with open(f"{model_dir}/label_encoder.pkl", "rb") as f: le = pickle.load(f)
        with open(f"{model_dir}/metadata.pkl", "rb") as f: meta = pickle.load(f)
        return model, scaler, le, meta
    else:
        return _train_fallback_model(df_raw)


In [ ]:
if df_raw is not None:
    model, scaler, le, meta = load_model(df_raw)
    print(f"✅ Modelo pronto: {meta.get('best_model_name')}")
    if meta.get("accuracy"):
        print(f"   Acurácia: {meta['accuracy']*100:.2f}%")
else:
    model = scaler = le = meta = None
    print("⚠️ Modelo não carregado — dados indisponíveis.")


## 5. Função de predição

A função `predict_obesity` recebe um dicionário com os dados do paciente, aplica a mesma engenharia de atributos do treinamento, escalona os dados (se aplicável) e retorna:
- a **classe predita** (nível de obesidade);
- as **probabilidades** para cada uma das 7 classes (quando o modelo suporta `predict_proba`, como é o caso do LightGBM).


In [ ]:
def predict_obesity(inputs: dict, model, scaler, le, meta) -> tuple:
    input_df = pd.DataFrame([inputs])
    X = engineer_features(input_df, meta["feature_columns"])
    X_proc = scaler.transform(X) if meta.get("scaled", False) else X.values
    pred_idx = model.predict(X_proc)[0]
    pred_class = le.inverse_transform([pred_idx])[0]
    proba = None
    if hasattr(model, "predict_proba"):
        proba_arr = model.predict_proba(X_proc)[0]
        proba = {le.inverse_transform([i])[0]: float(p) for i, p in enumerate(proba_arr)}
    return pred_class, proba


def hex_to_rgba(hex_color: str, alpha: float) -> str:
    """Converte cor hex para string rgba válida para o Plotly."""
    h = hex_color.lstrip("#")
    r, g, b = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
    return f"rgba({r},{g},{b},{alpha})"


### 5.1. Exemplo de predição

No app original, esses dados vinham de um **formulário interativo** (Streamlit).
Aqui, simulamos um paciente através de um dicionário Python — **edite os valores abaixo para testar outros perfis**.


In [ ]:
# Exemplo de paciente — altere os valores livremente
paciente_exemplo = dict(
    Gender="Female",
    Age=25,
    Height=1.70,
    Weight=70.0,
    family_history="yes",
    FAVC="yes",
    FCVC=2,
    NCP=3,
    CAEC="Sometimes",
    SMOKE="no",
    CH2O=2,
    SCC="no",
    FAF=1,
    TUE=1,
    CALC="Sometimes",
    MTRANS="Public_Transportation",
)

if model is not None:
    bmi_calc = paciente_exemplo["Weight"] / (paciente_exemplo["Height"] ** 2)
    pred_class, proba = predict_obesity(paciente_exemplo, model, scaler, le, meta)

    display(Markdown(f"### 🩺 Diagnóstico preditivo: **{LABEL_PT[pred_class]}**"))
    print(f"IMC calculado: {bmi_calc:.1f} kg/m²")
else:
    print("⚠️ Execute as células de carregamento de dados e modelo primeiro.")


### 5.2. Visualização do resultado

Reproduzimos os três gráficos usados no app para explicar visualmente o resultado da predição:
1. **Barra horizontal** de probabilidades por classe.
2. **Gauge (velocímetro)** de IMC, com faixas de referência da OMS.
3. **Radar** do perfil comportamental do paciente (risco x hábitos saudáveis x atividade física x hidratação x dieta).


In [ ]:
if model is not None and proba is not None:
    # 1) Probabilidades por classe
    prob_df = pd.DataFrame([
        {"Classe": LABEL_PT[k], "Probabilidade": v}
        for k, v in sorted(proba.items(), key=lambda x: ORDER.index(x[0]))
    ])
    color_list = [COLOR_MAP[k] for k in sorted(proba.keys(), key=lambda x: ORDER.index(x))]
    fig_prob = px.bar(
        prob_df, x="Probabilidade", y="Classe", orientation="h",
        color="Classe", color_discrete_sequence=color_list,
        text=prob_df["Probabilidade"].apply(lambda v: f"{v*100:.1f}%"),
        height=320, title="Probabilidade por Classe",
    )
    fig_prob.update_layout(
        showlegend=False,
        yaxis=dict(categoryorder="array", categoryarray=[LABEL_PT[c] for c in ORDER]),
    )
    fig_prob.update_traces(textposition="outside")
    fig_prob.show()


In [ ]:
COLOR_MAP_BAR = {
    "Insufficient_Weight": "#388e3c",
    "Normal_Weight":       "#689f38",
    "Overweight_Level_I":  "#fbc02d",
    "Overweight_Level_II": "#f57c00",
    "Obesity_Type_I":      "#e64a19",
    "Obesity_Type_II":     "#c62828",
    "Obesity_Type_III":    "#8e0000",
}

if model is not None:
    # 2) Gauge de IMC
    fig_gauge = go.Figure(go.Indicator(
        mode="gauge+number",
        value=round(bmi_calc, 1),
        title={"text": "IMC do Paciente"},
        gauge={
            "axis": {"range": [10, 50]},
            "bar": {"color": COLOR_MAP_BAR[pred_class]},
            "steps": [
                {"range": [10, 18.5], "color": "#acd6ae"},
                {"range": [18.5, 25], "color": "#c8df7a"},
                {"range": [25, 30],   "color": "#fff38a"},
                {"range": [30, 35],   "color": "#ffc96b"},
                {"range": [35, 40],   "color": "#ffa285"},
                {"range": [40, 50],   "color": "#ec8482"},
            ],
            "threshold": {"line": {"color": "black", "width": 3}, "thickness": 0.9, "value": bmi_calc},
        },
        number={"suffix": " kg/m²", "font": {"size": 28}},
    ))
    fig_gauge.update_layout(height=280, margin=dict(l=20, r=20, t=40, b=10))
    fig_gauge.show()


In [ ]:
if model is not None:
    # 3) Radar de perfil comportamental
    favc, family_history = paciente_exemplo["FAVC"], paciente_exemplo["family_history"]
    smoke, scc = paciente_exemplo["SMOKE"], paciente_exemplo["SCC"]
    caec, calc = paciente_exemplo["CAEC"], paciente_exemplo["CALC"]
    faf, fcvc, ch2o, mtrans = paciente_exemplo["FAF"], paciente_exemplo["FCVC"], paciente_exemplo["CH2O"], paciente_exemplo["MTRANS"]

    risk = (
        (favc == "yes") * 2 + (family_history == "yes") * 2 + (smoke == "yes") +
        (scc == "no") + (caec == "Always") * 2 + (caec == "Frequently") +
        (calc == "Always") * 2 + (calc == "Frequently")
    )
    healthy = (
        (faf >= 2) * 2 + (fcvc >= 2) + (ch2o >= 2) + (scc == "yes") +
        (mtrans in ["Walking", "Bike"])
    )
    categories = ["Risco<br>Comportamental", "Hábitos<br>Saudáveis", "Ativ.<br>Física", "Hidratação", "Dieta"]
    values = [min(risk / 10, 1), min(healthy / 6, 1), faf / 3, (ch2o - 1) / 2, (fcvc - 1) / 2]

    line_color = COLOR_MAP[pred_class]
    fill_color = hex_to_rgba(COLOR_MAP[pred_class], 0.35)

    fig_radar = go.Figure()
    fig_radar.add_trace(go.Scatterpolar(
        r=values + [values[0]], theta=categories + [categories[0]],
        fill="toself", fillcolor=fill_color, line=dict(color=line_color, width=2),
        name="Paciente",
    ))
    fig_radar.update_layout(
        polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
        showlegend=False, height=280, title="Perfil do Paciente",
    )
    fig_radar.show()

    print("\n💊 Recomendações Clínicas:")
    for rec in RECOMMENDATIONS[pred_class]:
        print(f"  ▶ {rec}")


## 6. Dashboard Analítico

Esta segunda parte reproduz a página **"Dashboard Analítico"** do app, que explora a base completa de pacientes (`Obesity.csv`) para identificar padrões relacionados à obesidade.

No app original, existiam **filtros interativos** (gênero, faixa etária, nível de obesidade) via widgets do Streamlit. Aqui, definimos os filtros como variáveis Python simples — altere-as para refazer os gráficos com subconjuntos diferentes dos dados.


In [ ]:
if df_raw is not None:
    df_dash = df_raw.copy()
    df_dash["BMI"] = df_dash["Weight"] / (df_dash["Height"] ** 2)
    df_dash["Obesity_PT"] = df_dash["Obesity"].map(LABEL_PT)

    # Filtros (edite livremente)
    gen_filter = ["Female", "Male"]
    age_filter = (14, 61)
    cls_filter = ORDER

    df_f = df_dash[
        df_dash["Gender"].isin(gen_filter) &
        df_dash["Age"].between(*age_filter) &
        df_dash["Obesity"].isin(cls_filter)
    ]
    print(f"{fmt_num(len(df_f))} registros após filtros aplicados.")


### 6.1. KPIs gerais

Indicadores-resumo: total de pacientes, percentual com obesidade, percentual com sobrepeso, percentual com peso normal e IMC médio.


In [ ]:
if df_raw is not None:
    total     = len(df_dash)
    obesos    = df_dash["Obesity"].isin(["Obesity_Type_I", "Obesity_Type_II", "Obesity_Type_III"]).sum()
    sobrepeso = df_dash["Obesity"].isin(["Overweight_Level_I", "Overweight_Level_II"]).sum()
    normal    = df_dash["Obesity"].isin(["Normal_Weight"]).sum()
    imc_medio = df_dash["BMI"].mean()

    kpis = pd.DataFrame({
        "Indicador": ["Total de Pacientes", "Com Obesidade", "Sobrepeso", "Peso Normal", "IMC Médio (kg/m²)"],
        "Valor": [
            fmt_num(total),
            f"{fmt_num(obesos)} ({obesos/total*100:.0f}%)",
            f"{fmt_num(sobrepeso)} ({sobrepeso/total*100:.0f}%)",
            f"{fmt_num(normal)} ({normal/total*100:.0f}%)",
            f"{imc_medio:.1f}",
        ],
    })
    display(kpis)


### 6.2. Distribuição e perfil demográfico

In [ ]:
if df_raw is not None:
    dist = df_f["Obesity"].value_counts().reindex(ORDER).fillna(0).reset_index()
    dist.columns = ["Obesity", "Quantidade"]
    dist["Nível"] = dist["Obesity"].map(LABEL_PT)
    fig1 = px.bar(
        dist, x="Nível", y="Quantidade", color="Obesity", color_discrete_map=COLOR_MAP,
        text="Quantidade", title="Distribuição por Nível de Obesidade",
        labels={"Nível": "", "Quantidade": "Pacientes"},
    )
    fig1.update_traces(textposition="outside")
    fig1.update_layout(showlegend=False, margin=dict(t=40, b=60))
    fig1.update_xaxes(tickangle=-30)
    fig1.update_yaxes(showgrid=False, showticklabels=False)
    fig1.show()


In [ ]:
if df_raw is not None:
    gd = df_f.groupby(["Obesity", "Gender"]).size().reset_index(name="Quantidade")
    gd["Nível"] = gd["Obesity"].map(LABEL_PT)
    gd["Gênero"] = gd["Gender"].map({"Female": "Feminino", "Male": "Masculino"})
    fig2 = px.bar(
        gd, x="Nível", y="Quantidade", color="Gênero", barmode="group",
        title="Distribuição por Gênero e Nível de Obesidade",
        labels={"Nível": "", "Quantidade": "Pacientes"},
        color_discrete_map={"Feminino": COLOR_F, "Masculino": COLOR_M},
    )
    fig2.update_layout(margin=dict(t=40, b=60))
    fig2.update_xaxes(tickangle=-30)
    fig2.show()


### 6.3. IMC e idade por nível de obesidade

In [ ]:
if df_raw is not None:
    bmi_box = df_f.copy()
    bmi_box["Nível"] = bmi_box["Obesity"].map(LABEL_PT)
    COLOR_MAP_DASH_PT = {LABEL_PT[k]: v for k, v in COLOR_MAP_DASH.items()}
    fig3 = px.box(
        bmi_box, x="Nível", y="BMI", color="Nível", color_discrete_map=COLOR_MAP_DASH_PT,
        title="Distribuição do IMC por Nível de Obesidade",
        labels={"Nível": "", "BMI": "IMC (kg/m²)"},
        category_orders={"Nível": [LABEL_PT[o] for o in ORDER]},
    )
    fig3.update_layout(showlegend=False, margin=dict(t=40, b=60))
    fig3.update_xaxes(tickangle=-30)
    fig3.show()


In [ ]:
if df_raw is not None:
    sc_df = df_f.copy()
    sc_df["Nível"] = sc_df["Obesity"].map(LABEL_PT)
    fig4 = px.scatter(
        sc_df.sample(min(600, len(sc_df)), random_state=42),
        x="Age", y="BMI", color="Nível", color_discrete_map=COLOR_MAP_DASH_PT,
        opacity=0.65, title="Relação Idade × IMC",
        labels={"Age": "Idade (anos)", "BMI": "IMC", "Nível": "Nível"},
        hover_data=["Weight", "Height"],
        category_orders={"Nível": [LABEL_PT[o] for o in ORDER]},
    )
    fig4.update_layout(legend_title="Nível", margin=dict(t=40))
    fig4.show()


### 6.4. Fatores de risco e hábitos de vida

In [ ]:
if df_raw is not None:
    fp = (
        df_f.groupby("Obesity")["family_history"]
        .apply(lambda x: (x == "yes").mean() * 100)
        .reindex(ORDER).reset_index()
    )
    fp.columns = ["Obesity", "Percentual"]
    fp["Nível"] = fp["Obesity"].map(LABEL_PT)
    fig5 = px.bar(
        fp, x="Nível", y="Percentual", color="Obesity", color_discrete_map=COLOR_MAP,
        title="% com Histórico Familiar", labels={"Nível": "", "Percentual": ""},
        text=fp["Percentual"].round(1).astype(str) + "%",
    )
    fig5.update_layout(showlegend=False, margin=dict(t=40, b=60))
    fig5.update_xaxes(tickangle=-30)
    fig5.update_yaxes(showgrid=False, showticklabels=False)
    fig5.show()


In [ ]:
if df_raw is not None:
    fm = df_f.groupby("Obesity")["FAF"].mean().reindex(ORDER).reset_index()
    fm["Nível"] = fm["Obesity"].map(LABEL_PT)
    fig6 = px.bar(
        fm, x="Nível", y="FAF", color="Obesity", color_discrete_map=COLOR_MAP,
        title="Frequência Média de Atividade Física", labels={"Nível": "", "FAF": "FAF médio (0–3)"},
        text=fm["FAF"].round(2),
    )
    fig6.update_traces(textposition="outside")
    fig6.update_layout(showlegend=False, margin=dict(t=40, b=60))
    fig6.update_xaxes(tickangle=-30)
    fig6.update_yaxes(showgrid=False, showticklabels=False)
    fig6.show()


In [ ]:
if df_raw is not None:
    fc = (
        df_f.groupby("Obesity")["FAVC"]
        .apply(lambda x: (x == "yes").mean() * 100)
        .reindex(ORDER).reset_index()
    )
    fc.columns = ["Obesity", "Percentual"]
    fc["Nível"] = fc["Obesity"].map(LABEL_PT)
    fig7 = px.bar(
        fc, x="Nível", y="Percentual", color="Obesity", color_discrete_map=COLOR_MAP,
        title="% que Consome Alimentos Calóricos", labels={"Nível": "", "Percentual": ""},
        text=fc["Percentual"].round(1).astype(str) + "%",
    )
    fig7.update_layout(showlegend=False, margin=dict(t=40, b=60))
    fig7.update_xaxes(tickangle=-30)
    fig7.update_yaxes(showgrid=False, showticklabels=False)
    fig7.show()


### 6.5. Transporte e correlação entre variáveis

In [ ]:
if df_raw is not None:
    mt = df_f[df_f["Obesity"].str.contains("Obesity")].copy()
    mt["Transporte"] = mt["MTRANS"].map({
        "Public_Transportation": "Transp. Público", "Walking": "A pé",
        "Automobile": "Automóvel", "Motorbike": "Moto", "Bike": "Bicicleta",
    })
    fig8 = px.histogram(mt, x="Transporte", text_auto=True, title="Obesidade por tipos de transporte")
    fig8.update_traces(marker_color="green")
    fig8.update_layout(xaxis_title="", yaxis_title="", margin=dict(t=50))
    fig8.update_yaxes(showgrid=False, showticklabels=False)
    fig8.show()


In [ ]:
if df_raw is not None:
    num_cols = ["Age", "BMI", "FCVC", "NCP", "CH2O", "FAF", "TUE"]
    corr_df = df_f[num_cols].corr().round(2)
    fig9 = px.imshow(
        corr_df, text_auto=True, color_continuous_scale=["#e8f5e9", "#a5d6a7", "#2e7d32"],
        zmin=-1, zmax=1, title="Correlação entre Variáveis Numéricas", aspect="auto",
    )
    fig9.update_layout(margin=dict(t=50))
    fig9.show()


### 6.6. Principais insights para a equipe médica

Por fim, o app calcula e apresenta um conjunto de **insights automáticos** a partir da própria base de dados: papel do histórico familiar, impacto do sedentarismo, alimentação calórica, monitoramento calórico, transporte passivo e hidratação.


In [ ]:
if df_raw is not None:
    obesos_mask = df_f["Obesity"].isin(["Obesity_Type_I", "Obesity_Type_II", "Obesity_Type_III"])
    pct_fam   = df_f[obesos_mask]["family_history"].value_counts(normalize=True).get("yes", 0) * 100
    faf_ob3   = df_f[df_f["Obesity"] == "Obesity_Type_III"]["FAF"].mean()
    faf_norm  = df_f[df_f["Obesity"] == "Normal_Weight"]["FAF"].mean()
    pct_favc  = df_f[obesos_mask]["FAVC"].value_counts(normalize=True).get("yes", 0) * 100
    pct_scc   = df_f[df_f["SCC"] == "yes"].shape[0] / len(df_f) * 100
    pct_trans = df_f[df_f["MTRANS"].isin(["Automobile", "Motorbike"])].shape[0] / len(df_f) * 100
    ch2o_norm = df_f[df_f["Obesity"] == "Normal_Weight"]["CH2O"].mean()
    ch2o_ob3  = df_f[df_f["Obesity"] == "Obesity_Type_III"]["CH2O"].mean()

    insights = [
        ("🧬 Fator Genético",
         f"{pct_fam:.0f}% dos pacientes obesos possuem histórico familiar de excesso de peso — "
         f"o fator hereditário é determinante no risco."),
        ("🏃 Sedentarismo",
         f"Pacientes com Obesidade Tipo III praticam em média {faf_ob3:.1f}/3 de atividade física "
         f"semanal, contra {faf_norm:.1f}/3 no grupo de Peso Normal."),
        ("🍔 Alimentação Calórica",
         f"{pct_favc:.0f}% dos pacientes obesos consomem alimentos altamente calóricos com frequência."),
        ("📊 Monitoramento Calórico",
         f"Apenas {pct_scc:.1f}% dos pacientes monitoram sua ingestão calórica — intervenção "
         f"educativa de alto impacto potencial."),
        ("🚗 Transporte Passivo",
         f"{pct_trans:.0f}% dos pacientes utilizam transporte motorizado, reduzindo a atividade "
         f"física diária."),
        ("💧 Hidratação",
         f"O consumo adequado de água é maior no grupo de Peso Normal ({ch2o_norm:.1f}/3) em "
         f"comparação à Obesidade Tipo III ({ch2o_ob3:.1f}/3)."),
    ]

    for title, text in insights:
        display(Markdown(f"**{title}** — {text}"))


## 7. Conclusão

Este notebook demonstrou, de forma explicativa, como o aplicativo Streamlit `app.py` do **Tech Challenge – Fase 04** funciona por dentro:

- Como os **dados brutos** são carregados e transformados (`engineer_features`);
- Como o **modelo LightGBM** é carregado (ou treinado como fallback) e usado para prever o nível de obesidade de um paciente;
- Como as **visualizações analíticas** revelam relações entre hábitos de vida e obesidade na base de dados.

### ▶️ Para rodar a versão web completa (com formulário interativo e navegação entre páginas):

```bash
pip install streamlit pandas numpy scikit-learn xgboost lightgbm matplotlib seaborn plotly
streamlit run app.py
```
